# Old API test

## SetUp



In [1]:
import requests
clientSlug = 'experiment'
username = 'frecognition'
password ='Frecog2025@'
api_host = "https://smart-office-api.humblebee.ai/"

create_user_api_url = api_host + "api/:{clientSlug}/history/create"
get_all_users_api_url = api_host + 'api/{clientSlug}/users'


In [ ]:
base_url = "https://smart-office-api.humblebee.ai/api"
session = requests.Session()

In [ ]:
from typing import Optional, Dict, Any
def login(username: str, password: str, client_slug: str) -> Dict[str, Any]:
        """
        Authenticate user and obtain access token

        Args:
            username (str): User's username
            password (str): User's password
            client_slug (str): Client slug for organization

        Returns:
            dict: Login response data

        Raises:
            requests.exceptions.RequestException: If login fails
        """
        login_data = {
            "username": username,
            "password": password,
            "client_slug": client_slug
        }

        try:
            response = session.post(
                f"{base_url}/auth/login",
                json=login_data,
                headers={"Content-Type": "application/json"}
            )
            response.raise_for_status()

            data = response.json()

            if data.get('success'):
                token = data.get('token')
                client_slug = client_slug
                # Set authorization header for future requests
                session.headers.update({
                    'Authorization': f'Bearer {token}'
                })
                return data, token, client_slug
            else:
                raise requests.exceptions.RequestException(f"Login failed: {data.get('error', 'Unknown error')}")

        except requests.exceptions.RequestException as e:
            raise requests.exceptions.RequestException(f"Login request failed: {str(e)}")

In [ ]:
login_response, token, client_slug = login(username, password, clientSlug)
print(login_response)
token = login_response.get('token')

## Send Unrecognized Face

In [ ]:
import requests

url = base_url + f'/{client_slug}/unrecognized'
print(url)
files = [
    ('images', ('Doston_Sayfiddinov_4.jpg', open('Sample_4.jpg', 'rb'), 'image/jpeg')),
]


data = {
    'user_status': 'IN',
    'notes': 'Unknown person spotted',
}

headers = {
    'Authorization': f'Bearer {token}'
}

response = requests.post(url, headers=headers, files=files, data=data)
print(response.json())


## Get users and Create record

In [ ]:
def create_record(self, user_id: int, status: str) -> Dict[str, Any]:
        """
        Create an attendance record

        Args:
            user_id (int): ID of the user to create record for
            status (str): Either 'in' or 'out'

        Returns:
            dict: Record creation response data

        Raises:
            requests.exceptions.RequestException: If record creation fails
        """
        if not self.token:
            raise ValueError("Not authenticated. Please login first.")
        status = status.lower()
        if status not in ['in', 'out']:
            raise ValueError("Status must be either 'in' or 'out'")

        record_data = {
            "user_id": user_id,
            "status": status
        }

        try:
            response = self.session.post(
                f"{self.base_url}/{self.client_slug}/history/create",
                json=record_data,
                headers={"Content-Type": "application/json"}
            )
            response.raise_for_status()

            return response

        except requests.exceptions.RequestException as e:
            raise requests.exceptions.RequestException(f"Record creation request failed: {str(e)}")


In [ ]:
def get_users(page: int = 1, limit: int = 50) -> Dict[str, Any]:
        """
        Get list of users for the current client

        Args:
            page (int): Page number for pagination
            limit (int): Number of users per page

        Returns:
            dict: Users list response data

        Raises:
            requests.exceptions.RequestException: If request fails
        """
        if not token:
            raise ValueError("Not authenticated. Please login first.")

        try:
            response = session.get(
                f"{base_url}/{client_slug}/users",
                params={"page": page, "limit": limit, "status": "active"}
            )
            response.raise_for_status()

            data = response.json()

            # Handle different response formats
            if isinstance(data, dict):
                # If it's a dict with success field
                if not data.get('success', True):
                    raise requests.exceptions.RequestException(f"Failed to get users: {data.get('error', 'Unknown error')}")
                return data
            elif isinstance(data, list):
                # If it's a direct list of users
                return {"success": True, "users": data}
            else:
                # If it's some other format, try to return as is
                return {"success": True, "users": data}

        except requests.exceptions.RequestException as e:
            raise requests.exceptions.RequestException(f"Get users request failed: {str(e)}")

In [ ]:
user_responce = get_users()
users_data = user_responce.get("users", [28090])


In [ ]:
from typing import Dict, Any
def create_record(user_id: int, status: str) -> Dict[str, Any]:
    """
    Create an attendance record

    Args:
        user_id (int): ID of the user to create record for
        status (str): Either 'in' or 'out'

    Returns:
        dict: Record creation response data

    Raises:
        requests.exceptions.RequestException: If record creation fails
    """
    if not token:
        raise ValueError("Not authenticated. Please login first.")
    status = status.lower()
    if status not in ['in', 'out']:
        raise ValueError("Status must be either 'in' or 'out'")

    record_data = {
        "user_id": user_id,
        "status": status
    }

    try:
        response = session.post(
            f"{base_url}/{client_slug}/history/create",
            json=record_data,
            headers={"Content-Type": "application/json"}
        )
        response.raise_for_status()


        data = response.json()

        if not data.get('success'):
            raise requests.exceptions.RequestException(f"Record creation failed: {data.get('error', 'Unknown error')}")

        return data

    except requests.exceptions.RequestException as e:
        raise requests.exceptions.RequestException(f"Record creation request failed: {str(e)}")


In [ ]:
responce = create_record(56574, 'OUT')
print(responce)


## Get images from Dasdhboard

## get last status of all users


In [ ]:
#use /api/:clientSlug/history api
def get_history(page: int = 1, limit: int = 50) -> Dict[str, Any]:
    """
    Get attendance history for the current client

    Args:
        page (int): Page number for pagination
        limit (int): Number of records per page

    Returns:
        dict: History response data

    Raises:
        requests.exceptions.RequestException: If request fails
    """
    if not token:
        raise ValueError("Not authenticated. Please login first.")

    try:
        response = session.get(
            f"{base_url}/{client_slug}/history",
            params={"page": page, "limit": limit}
        )
        response.raise_for_status()

        data = response.json()

        if not data.get('success', True):
            raise requests.exceptions.RequestException(f"Failed to get history: {data.get('error', 'Unknown error')}")

        return data

    except requests.exceptions.RequestException as e:
        raise requests.exceptions.RequestException(f"Get history request failed: {str(e)}")

In [ ]:
get_history()

# New Api


In [2]:
!pip install dotenv

  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
Using cached python_dotenv-1.1.1-py3-none-any.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dotenv]


In [32]:
## get token

import os
import requests
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()  # reads .env if present

BASE_URL = os.getenv("SO_BACKEND_API_URL", "http://localhost:7091/api")
EMAIL    = os.getenv("SO_SUPERADMIN_EMAIL", "admin@humblebee.ai")
PASSWORD = os.getenv("SO_SUPERADMIN_PASSWORD", "SM_SUPERADMIN_PASSWORD123!")

session = requests.Session()
session.headers.update({"Content-Type": "application/json", "accept": "application/json"})

def login():
    r = session.post(f"{BASE_URL}/auth/login", json={"email": EMAIL, "password": PASSWORD})
    r.raise_for_status()
    data = r.json()
    token = data.get("token")
    if not token:
        raise RuntimeError(f"No token in login response: {data}")
    session.headers.update({"Authorization": f"Bearer {token}"})
    print("✅ Logged in as", data.get("user", {}).get("email"))
    return data

_ = login()


✅ Logged in as admin@humblebee.ai


In [33]:
## Get users
def list_org_users(slug: str, page: int = 1, limit: int = 50):
    r = session.get(f"{BASE_URL}/org/{slug}/users", params={"page": page, "limit": limit})
    r.raise_for_status()
    return r.json()

slug = "dev"  # change to your org slug

users = list_org_users(slug)
print(f"Found {len(users)} users in '{slug}' (showing first 5)")
# print(users[0])
for u in users[:5]:#customize
    print(u.get("id"), u.get("full_name"), u.get("email"))


Found 53 users in 'dev' (showing first 5)
59 Antvision saxdev@gmail.com
56 Asadbek Otabekov dev@gmail.com
51 Oybek Inokov 
4 Foinn Igztsxg foinn.igztsxg@dev.org
5 Quncl Udwegfw quncl.udwegfw@dev.org


In [34]:
##Camera
def get_cameras(slug):
    r = session.get(f"{BASE_URL}/org/{slug}/cameras")
    r.raise_for_status()
    return r.json()

# users = get_users(slug)
cameras = get_cameras(slug)
print(f"Found {len(users)} users and {len(cameras)} cameras")
camera_id = cameras[0]["id"]
# if users: print("Sample user:", users[0])
if cameras: print("Sample camera:", camera_id)


Found 53 users and 6 cameras
Sample camera: 1


In [43]:
# Attandasce record
user = users[0]  # first user in the list
user_id = user["id"]

camera_id = 1  # or whatever camera ID you want
slug = "dev"

def iso_now():
    import datetime
    return datetime.datetime.utcnow().isoformat() + "Z"

payload = {
    "user_id": int(51),
    "status": "in",          # or "out"
    "camera_id": int(camera_id),
    "timestamp": iso_now()
}

print(f'payload: {payload}')

r = session.post(f"{BASE_URL}/org/{slug}/attendance-records", json=payload,headers={"Content-Type": "application/json"})
r.raise_for_status()

result = r.json()
print(result)
print("✅ Attendance record created:")
print(f"   ID: {result.get('id')}")
print(f"   External ID: {result.get('external_id')}")
print(f"   External Link: {result.get('external_link')}")
print(f"   Status: {result.get('status')}")
print(f"   Timestamp: {result.get('timestamp')}")

payload: {'user_id': 51, 'status': 'in', 'camera_id': 1, 'timestamp': '2025-10-26T10:15:38.057397Z'}
{'created_at': '2025-10-26T10:15:38.069Z', 'id': '2383', 'user_id': '51', 'status': 'in', 'camera_id': '1', 'timestamp': '2025-10-26T10:15:38.057Z', 'external_id': 'bf467a96-07c8-49a1-8161-3d20997fd7e8'}
✅ Attendance record created:
   ID: 2383
   External ID: bf467a96-07c8-49a1-8161-3d20997fd7e8
   External Link: None
   Status: in
   Timestamp: 2025-10-26T10:15:38.057Z


In [27]:
import json

# Get user_id from the earlier cell
user = users[0]  # first user in the list
user_id = user["id"]

payload = {
    "user_id": int(user_id),          # keep original type
    "status": "out",              # adjust if your enum differs
    "camera_id": int(camera_id),      # keep as '2' (string) per your sample
    "timestamp": iso_now()
}
print(f'payload: {payload}')

r = session.post(f"{BASE_URL}/org/{slug}/attendance-records", json=payload)
print("HTTP", r.status_code)
try:
    print(json.dumps(r.json(), indent=2))
except Exception:
    print(r.text[:1200])
r.raise_for_status()

payload: {'user_id': 59, 'status': 'out', 'camera_id': 1, 'timestamp': '2025-10-26T08:11:22.302664Z'}
HTTP 500
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Error</title>
</head>
<body>
<pre>Error<br> &nbsp; &nbsp;at Query.run (/app/node_modules/sequelize/lib/dialects/postgres/query.js:50:25)<br> &nbsp; &nbsp;at /app/node_modules/sequelize/lib/sequelize.js:315:28<br> &nbsp; &nbsp;at process.processTicksAndRejections (node:internal/process/task_queues:105:5)<br> &nbsp; &nbsp;at async PostgresQueryInterface.insert (/app/node_modules/sequelize/lib/dialects/abstract/query-interface.js:308:21)<br> &nbsp; &nbsp;at async model.save (/app/node_modules/sequelize/lib/model.js:2490:35)<br> &nbsp; &nbsp;at async AttendanceRecord.create (/app/node_modules/sequelize/lib/model.js:1362:12)<br> &nbsp; &nbsp;at async AttendanceRecordController.create (/app/src/controllers/AttendanceRecordController.js:68:20)</pre>
</body>
</html>



HTTPError: 500 Server Error: Internal Server Error for url: http://localhost:7091/api/org/dev/attendance-records

In [28]:
import requests
from typing import List, Dict, Any, Optional

def get_users_currently_in(
    base_url: str,
    slug: str,
    token: str,
    timeout: float = 10.0,
    verify_ssl: bool = True,
) -> Optional[List[Dict[str, Any]]]:
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json",
    }
    url = f"{BASE_URL}/api/org/{slug}/users/in"
    try:
        resp = requests.get(url, headers=headers, timeout=timeout, verify=verify_ssl)
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return None

    # Basic debugging info
    print(f"Status code: {resp.status_code}")

    if resp.status_code == 200:
        try:
            data = resp.json()
        except ValueError:
            print("Could not decode JSON.")
            return None

        # Pretty-print a quick summary
        print(f"Received {len(data)} user(s). Example:")
        if len(data) > 0:
            first = data[0]
            print(
                f"- id: {first.get('id')}, "
                f"username: {first.get('username')}, "
                f"status: {first.get('status')}, "
                f"branch_id: {first.get('branch_id')}"
            )
        return data
    else:
        # Show server response text for debugging
        print("Response body:")
        print(resp.text)
        return None
